In [1]:
import requests
import torch
import re
import time
import psutil
import subprocess

import pandas as pd

from datasets import load_dataset

In [2]:
ds = load_dataset("christinacdl/binary_hate_speech")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3883 entries, 0 to 3882
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    3883 non-null   object
 1   label   3883 non-null   object
dtypes: object(2)
memory usage: 60.8+ KB


In [3]:
test['label'] = test['label'].apply(lambda x: 'hateful' if x == 'OFF_HATEFUL_TOXIC' else 'safe')

labels = test['label'].unique()

test

,text,label
0,i have to study... #face #pizza (i stole my ...,safe
1,days porn movie srilankanboyssex,safe
2,feeling for friends left in the place we use...,safe
3,why only target little #muslim children for mi...,hateful
4,M. Todd Henderson dared compare SCOTUS nominee...,safe
...,...,...
3878,Maybe he just thought you looked like a sand n...,hateful
3879,ATTENTION SYRIAN 'REFUGEES'! You can go home n...,safe
3880,A hoe wants attention a women wants respect.,hateful
3881,@user there is only one requirement for the jo...,hateful


In [4]:
def get_ollama_memory_usage(port=11434):
    """
    Finds the process listening on the given port using psutil
    and returns its memory usage in bytes (RSS).
    Returns None if the process isn't found or can't be accessed.
    """
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            # Call proc.connections() to see if it's listening on the desired port
            for conn in proc.connections(kind='inet'):
                if conn.laddr.port == port:
                    # Found the process that listens on port=11434
                    memory_info = proc.memory_info()
                    return memory_info.rss  # in bytes
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            pass
    
    # If no process was found
    return None

get_ollama_memory_usage()

C:\Users\Rafael\AppData\Local\Temp\ipykernel_14168\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


96071680

In [5]:
def get_gpu_memory_usage():
    """
    Returns a list of used memory (in MB) for each GPU.
    """
    # Use nvidia-smi with the --query-gpu and --format flags to get just the memory usage
    command = [
        "nvidia-smi",
        "--query-gpu=memory.used",  # You can also add memory.free, name, etc.
        "--format=csv,noheader,nounits"  # CSV output with no header or units
    ]
    try:
        output = subprocess.check_output(command)
        # Decode the output from bytes to string
        output_str = output.decode("utf-8").strip()
        # Each line corresponds to one GPU's memory usage
        usage_values = [int(x) for x in output_str.split("\n")]
        return usage_values[0]
    except subprocess.CalledProcessError as e:
        print("Error running nvidia-smi:", e)
        return []


In [6]:
def classify(text, labels):
    url = "http://localhost:11434/api/chat"
    
    messages = [
        {"role": "system", "content": "You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling binary classification tasks based on user instructions."},
        {"role": "user", "content": f"Classify the following text based on the task: Sentiment analysis of possibly hateful content. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Tweet: {text}"}
    ]
    
    start_time = time.time()

    try:
        response = requests.post(url, json={
            "model": "deepseek-r1:1.5b",
            "messages": messages,
            "think": True,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 3100
            }
        }, timeout=30)
        response_time = time.time() - start_time
        vram_usage = get_gpu_memory_usage()
        ram_usage_bytes = get_ollama_memory_usage(port=11434) / (1024 * 1024)
        response = response.json()
        response_text = response['message'].get('thinking', '') if 'message' in response else ''
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return "error", {}, 0, 0, 0, 0, f"API Error: {e}"
    
    if not response.get('done', False):
        print(f"Ollama returned an incomplete response: {response.get('error')}")
        return 'error', {}, response_time, 0, 0, 0, response.get('error', 'Incomplete response')
    
    if 'message' in response and 'content' in response['message']:
        classification_text = response['message']['content'].lower()
        print("Response fields:", ', '.join(response.keys()))
        print(response)
        total_time = response['total_duration'] / 1_000_000_000
    else:
        messages.append({"role": "assistant", "content": response_text + '</think>'})
        start_time2 = time.time()
        response2 = requests.post(url, json={
            "model": "deepseek-r1:1.5b",
            "messages": messages,
            "think": False,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 430
            }
        })
        response_time += time.time() - start_time2
        response2 = response2.json()
        classification_text = response2['message']['content'].lower() if 'message' in response2 and 'content' in response2['message'] else ''
        total_time = response['total_duration'] / 1_000_000_000 + response2['total_duration'] / 1_000_000_000
    
    label_counts = {label: len(re.findall(r'\b' + re.escape(label.lower()) + r'\b', classification_text)) for label in labels}
    
    if all(count == label_counts[labels[0]] for count in label_counts.values()):
        content = 'error'
    else:
        content = max(label_counts, key=label_counts.get)
    
    print(f"Text: {text}")
    print(f"Response: {content}")
    
    return content, label_counts, response_time, vram_usage, ram_usage_bytes, total_time, response_text

In [7]:
# apply the classify function to the test set. create one column for each output
test[['prediction', 'label_counts','response_time', 'vram_usage', 'ram_usage', 'total_time', 'response_text']] = test['text'].apply(lambda x: classify(x, labels)).apply(pd.Series)

C:\Users\Rafael\AppData\Local\Temp\ipykernel_14168\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


Response fields: model, created_at, message, done_reason, done, total_duration, load_duration, prompt_eval_count, prompt_eval_duration, eval_count, eval_duration
{'model': 'deepseek-r1:1.5b', 'created_at': '2025-06-08T21:58:47.1711644Z', 'message': {'role': 'assistant', 'content': 'hateful', 'thinking': 'Okay, so I need to figure out how to classify this text into either "safe" or "hateful." Let me read through the text again carefully.\n\nThe text starts with "i have to study..." which seems like a normal sentence. Then it says "#face #pizza (i stole my roommate\'s idea for taking this pic)." Hmm, that looks like hashtags being used in a way where someone is stealing another person\'s idea and using it as their own. The use of hashtags might indicate some kind of manipulation or spreading of ideas.\n\nThe task is sentiment analysis of possibly hateful content. So I\'m thinking about whether this text could be considered hateful. The hashtags suggest that the speaker is taking credit f

In [8]:
test.to_csv('results/deepseekR1_ZS_binary1.csv', index=False)
test

,text,label,prediction,label_counts,response_time,vram_usage,ram_usage,total_time,response_text
0,i have to study... #face #pizza (i stole my ...,safe,hateful,"{'safe': 0, 'hateful': 1}",4.126712,2473,91.117188,2.070750,"Okay, so I need to figure out how to classify ..."
1,days porn movie srilankanboyssex,safe,safe,"{'safe': 1, 'hateful': 0}",3.966928,2483,91.480469,1.925953,"Okay, so I need to figure out how to classify ..."
2,feeling for friends left in the place we use...,safe,safe,"{'safe': 1, 'hateful': 0}",4.409216,2442,91.945312,2.358393,"Okay, so I need to figure out how to classify ..."
3,why only target little #muslim children for mi...,hateful,error,"{'safe': 0, 'hateful': 0}",3.948444,2462,92.113281,1.911032,"Okay, so I need to figure out whether this twe..."
4,M. Todd Henderson dared compare SCOTUS nominee...,safe,error,"{'safe': 0, 'hateful': 0}",4.494566,2464,92.128906,2.428690,"Alright, let's tackle this classification task..."
...,...,...,...,...,...,...,...,...,...
3878,Maybe he just thought you looked like a sand n...,hateful,error,"{'safe': 0, 'hateful': 0}",6.182457,2773,70.574219,4.130124,"Okay, so I need to figure out whether this twe..."
3879,ATTENTION SYRIAN 'REFUGEES'! You can go home n...,safe,hateful,"{'safe': 0, 'hateful': 1}",4.820069,2777,69.847656,2.786959,"Okay, so I need to classify this text into eit..."
3880,A hoe wants attention a women wants respect.,hateful,safe,"{'safe': 1, 'hateful': 0}",5.769430,2770,70.089844,3.728675,"Okay, so I need to figure out how to classify ..."
3881,@user there is only one requirement for the jo...,hateful,safe,"{'safe': 1, 'hateful': 0}",5.480467,2770,70.089844,3.431970,"Alright, let's break this down step by step. F..."


In [9]:
y_pred = test['prediction']
y_true = test['label']

#import acc, f1_score, precision and recall from sklearn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)

Accuracy: 0.535926
F1 score: 0.577700
Precision: 0.626614
Recall: 0.535926


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [10]:
# get average response time, vram usage and ram usage
response_time_avg = test['response_time'].mean()
vram_usage_avg = test['vram_usage'].mean()
ram_usage_avg = test['ram_usage'].mean()
total_time_avg = test['total_time'].mean()

print(f'Average response time: {response_time_avg}')
print(f'Average VRAM usage: {vram_usage_avg}')
print(f'Average RAM usage: {ram_usage_avg}')
print(f'Average total time: {total_time_avg}')

Average response time: 5.262651703451639
Average VRAM usage: 2441.013649240278
Average RAM usage: 77.04419706090651
Average total time: 3.214303923924801


In [11]:
# save results to txt
with open('results/deepseekR1_ZS_binary1.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {response_time_avg}\n')
    f.write(f'Average VRAM usage: {vram_usage_avg}\n')
    f.write(f'Average RAM usage: {ram_usage_avg}\n')
    f.write(f'Average total time: {total_time_avg}\n')
    f.write(f'Lines classified: {len(test)}\n')